# NorthStar — Initial Data Profiling

Quick look at the raw files: missing values, the zone naming problem, and the impossible timestamps in deliveries.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

RAW_DIR = Path('/content/drive/MyDrive/northstar-databases-analytics/data/raw')
print('Files found:', [f.name for f in RAW_DIR.glob('*.csv')])

customers = pd.read_csv(RAW_DIR / 'customers.csv')
deliveries = pd.read_csv(
    RAW_DIR / 'deliveries.csv',
    parse_dates=['dispatch_time', 'delivery_completed_at'],
)
orders = pd.read_csv(RAW_DIR / 'orders.csv')

datasets = {
    'customers': customers,
    'orders': orders,
}
print(f'Loaded {len(datasets)} datasets from explicit files')

## 2. Missing values heatmap

In [ ]:
missing = pd.DataFrame()
for name, df in datasets.items():
    missing[name] = df.isnull().mean() * 100
missing = missing.T
missing = missing.loc[:, (missing > 0).any()]
print(missing)

In [ ]:
plt.figure()
sns.heatmap(missing, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Missing (%)'})
plt.title('Missing Data Heatmap')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Zone naming inconsistency

The `home_zone` column has 16 raw values for only 7 actual zones.

In [ ]:
customers.groupby('home_zone').size().reset_index(name='total_records').sort_values('total_records', ascending=False)

## 4. Impossible timestamps in deliveries

Deliveries where `delivery_completed_at` is earlier than `dispatch_time` — operationally impossible. All are marked OnTime.

In [ ]:
deliveries.loc[deliveries['delivery_completed_at'] < deliveries['dispatch_time']] \
    .groupby('delivery_status') \
    .size() \
    .reset_index(name='count_early')

In [ ]:
deliveries['delivery_completed_at'].isnull().sum()

## 5. Missing counts per file

In [ ]:
for name in ['customers', 'orders']:
    print(f'--- {name} ---')
    print(datasets[name].isnull().sum())
    print()